# Multi-Agent Academic Research Assistant

This notebook implements the **Multi-Agent Academic Research Assistant** from the agent.md blueprint.

- **Tech stack**: CrewAI, Gradio
- **Flow**: User enters a research topic → Wikipedia → arXiv → Web Search → Synthesis → Report Writer → downloadable report

## 1. Install dependencies

In [ ]:
!pip install -q crewai crewai-tools gradio Wikipedia-API arxiv duckduckgo-search python-dotenv

## 2. Environment (optional)

Set `OPENAI_API_KEY` (or other LLM API key) in `.env` for the CrewAI agents. For web search we use DuckDuckGo (no key required).

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# CrewAI typically uses OPENAI_API_KEY for the default LLM
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("Warning: OPENAI_API_KEY not set. Set it in .env for full functionality.")

## 3. Custom tools

In [ ]:
from crewai.tools import tool
import wikipediaapi
import arxiv
from pathlib import Path


@tool("Wikipedia summary tool")
def wikipedia_tool(query: str) -> str:
    """Retrieve a high-level summary and key concepts for a research topic from Wikipedia."""
    try:
        wiki = wikipediaapi.Wikipedia(
            user_agent="AcademicResearchAssistant/1.0",
            language="en"
        )
        page = wiki.page(query)
        if not page.exists():
            # Try first sentence as summary for disambiguation
            return f"No Wikipedia page found for '{query}'. Try a more specific or alternative term."
        summary = page.summary
        sections = ""
        for section in page.sections[:5]:  # First 5 sections
            sections += f"\n## {section.title}\n{section.text[:1500]}"
        return f"# {page.title}\n\n{summary}{sections}"
    except Exception as e:
        return f"Wikipedia error: {e}"


@tool("arXiv papers tool")
def arxiv_tool(query: str) -> str:
    """Retrieve relevant academic papers from arXiv: title, authors, abstract, publication date."""
    try:
        client = arxiv.Client()
        search = arxiv.Search(
            query=query,
            max_results=5,
            sort_by=arxiv.SortCriterion.SubmittedDate
        )
        lines = []
        for i, result in enumerate(client.results(search), 1):
            lines.append(
                f"Paper {i}:\n"
                f"Title: {result.title}\n"
                f"Authors: {', '.join(a.name for a in result.authors)}\n"
                f"Date: {result.published}\n"
                f"Abstract: {result.summary}\n"
            )
        return "\n---\n".join(lines) if lines else f"No arXiv papers found for: {query}"
    except Exception as e:
        return f"arXiv error: {e}"


@tool("Web search tool")
def web_search_tool(query: str) -> str:
    """Perform a general web search for current developments, news, and insights on a topic."""
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=8))
        if not results:
            return f"No web results for: {query}"
        lines = []
        for i, r in enumerate(results, 1):
            title = r.get("title", "")
            body = r.get("body", "")
            href = r.get("href", "")
            lines.append(f"{i}. {title}\n{body}\nSource: {href}")
        return "\n\n".join(lines)
    except Exception as e:
        return f"Web search error: {e}"


# Report writer state (filename will be set by the crew run)
_report_output_dir = Path("./reports")
_report_output_dir.mkdir(exist_ok=True)


@tool("Report writer tool")
def report_writer_tool(content: str, filename: str) -> str:
    """Write the final research report to a file (Markdown). Creates the file in the reports/ directory."""
    try:
        path = _report_output_dir / filename
        if not filename.endswith(".md"):
            path = path.with_suffix(".md")
        path.write_text(content, encoding="utf-8")
        return f"Report saved to {path.absolute()}"
    except Exception as e:
        return f"Report write error: {e}"


print("Tools defined: wikipedia_tool, arxiv_tool, web_search_tool, report_writer_tool")

## 4. Agents and tasks (CrewAI)

In [ ]:
from crewai import Agent, Task, Crew, Process


def create_research_crew(topic: str):
    """Create agents and tasks for the research workflow."""
    
    # Wikipedia Research Agent
    wikipedia_agent = Agent(
        role="Wikipedia Research Agent",
        goal="Retrieve high-level conceptual understanding and key concepts from Wikipedia",
        backstory="You are an expert at summarizing encyclopedic knowledge and extracting key concepts.",
        tools=[wikipedia_tool],
        verbose=True,
        allow_delegation=False,
    )

    # Academic Paper Agent
    arxiv_agent = Agent(
        role="Academic Paper Agent",
        goal="Retrieve relevant peer-reviewed style academic papers from arXiv",
        backstory="You are skilled at finding and summarizing academic literature.",
        tools=[arxiv_tool],
        verbose=True,
        allow_delegation=False,
    )

    # Web Search Agent
    web_search_agent = Agent(
        role="Web Search Agent",
        goal="Gather current context, news, and real-time information from the web",
        backstory="You find recent developments, industry insights, and emerging trends.",
        tools=[web_search_tool],
        verbose=True,
        allow_delegation=False,
    )

    # Analysis & Synthesis Agent (no tools; uses context from previous tasks)
    synthesis_agent = Agent(
        role="Analysis & Synthesis Agent",
        goal="Transform collected research into structured knowledge and key insights",
        backstory="You compare sources, identify patterns and consensus, and organize information logically.",
        tools=[],
        verbose=True,
        allow_delegation=False,
    )

    # Report Writer Agent
    report_agent = Agent(
        role="Report Writer Agent",
        goal="Generate the final research report and save it to disk",
        backstory="You write clear, structured academic reports in Markdown.",
        tools=[report_writer_tool],
        verbose=True,
        allow_delegation=False,
    )

    # --- Tasks ---
    task_wikipedia = Task(
        description=f"Use wikipedia_tool to obtain a high-level summary and key concepts for: {topic}",
        agent=wikipedia_agent,
        expected_output="Topic overview, key concepts, and historical context from Wikipedia.",
    )

    task_arxiv = Task(
        description=f"Use arxiv_tool to retrieve relevant academic papers for: {topic}",
        agent=arxiv_agent,
        expected_output="List of relevant research papers with title, authors, abstract, and date.",
    )

    task_web = Task(
        description=f"Use web_search_tool to collect current developments and supplementary context for: {topic}",
        agent=web_search_agent,
        expected_output="News developments, industry insights, and recent breakthroughs from the web.",
    )

    task_synthesis = Task(
        description="""Analyze and synthesize all gathered information (Wikipedia, arXiv, web).
        Produce a structured outline with: Introduction, Background, Key Concepts, Important Research Papers,
        Recent Developments, Future Directions, and Conclusion. Do not write the full report yet—output the structured outline and key points.""",
        agent=synthesis_agent,
        expected_output="Structured outline and key points for the report.",
        context=[task_wikipedia, task_arxiv, task_web],
    )

    safe_name = "".join(c if c.isalnum() or c in " -_" else "_" for c in topic)[:50].strip().replace(" ", "_")
    report_filename = f"{safe_name}_research_report.md"

    task_report = Task(
        description=f"""Write the full research report in Markdown using the synthesis outline.
        Include: Introduction, Background, Key Concepts, Important Research Papers, Recent Developments, Future Directions, Conclusion.
        Then save the report using report_writer_tool with filename='{report_filename}'.""",
        agent=report_agent,
        expected_output="Final research report saved to file.",
        context=[task_synthesis],
    )

    crew = Crew(
        agents=[wikipedia_agent, arxiv_agent, web_search_agent, synthesis_agent, report_agent],
        tasks=[task_wikipedia, task_arxiv, task_web, task_synthesis, task_report],
        process=Process.sequential,
    )
    return crew, report_filename


print("Crew and task factory ready.")

## 5. Run research (single topic)

In [ ]:
def run_research(topic: str) -> str:
    """Execute the full research pipeline and return the report path and content."""
    crew, report_filename = create_research_crew(topic)
    result = crew.kickoff(inputs={})
    path = _report_output_dir / report_filename
    if path.exists():
        return path.read_text(encoding="utf-8"), str(path.absolute())
    return str(result), None


# Example (run in notebook):
# content, path = run_research("CRISPR gene editing")
# print(content[:2000])

## 6. Gradio interface

In [ ]:
import gradio as gr


def run_research_ui(topic: str):
    if not topic or not topic.strip():
        return "Please enter a research topic.", None
    try:
        content, path = run_research(topic.strip())
        return content, path if path else None
    except Exception as e:
        return f"Error: {e}", None


with gr.Blocks(title="Academic Research Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Multi-Agent Academic Research Assistant")
    gr.Markdown("Enter a research topic and click **Run Research** to generate a structured report (Wikipedia + arXiv + Web → synthesis → report).")
    with gr.Row():
        topic_in = gr.Textbox(
            label="Research topic",
            placeholder="e.g. CRISPR gene editing",
            scale=3,
        )
        run_btn = gr.Button("Run Research", variant="primary", scale=1)
    report_out = gr.Textbox(
        label="Research report",
        lines=20,
        max_lines=40,
    )
    file_out = gr.File(label="Download report")

    def on_run(topic):
        content, path = run_research_ui(topic)
        return content, path if path else None

    run_btn.click(
        fn=on_run,
        inputs=[topic_in],
        outputs=[report_out, file_out],
    )

demo.launch()